# Minimal modifiedTexas2k OPF

Load the MATPOWER case, apply load scenario 0, and solve one OPF.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pyarrow.dataset as ds
import juliapkg.deps as _juliapkg_deps

# gridfm-datakit compatibility with newer juliapkg
if not hasattr(_juliapkg_deps, "run_julia") and hasattr(_juliapkg_deps, "run_script"):
    def _run_julia_compat(script, *args, **kwargs):
        if os.name == "nt":
            script = [line.replace("\\", "/") if isinstance(line, str) else line for line in script]
        return _juliapkg_deps.run_script(script, *args, **kwargs)

    _juliapkg_deps.run_julia = _run_julia_compat

from gridfm_datakit.network import load_net_from_file
from gridfm_datakit.process.process_network import init_julia
from gridfm_datakit.process.solvers import run_opf


In [ ]:
data_dir = Path(r"C:\Users\Coham\OneDrive\Desktop\pyscripts\projects\Powermodels\data")
case_file = data_dir / "modifiedTexas2k.m"
load_file = data_dir / "modifiedTexas2k_loads_scenarios_0_23.parquet"

SCENARIO = 0
REFERENCE_SCENARIO = 16


In [ ]:
# Load network
network = load_net_from_file(str(case_file))

# Rows corresponding to load buses; MATPOWER columns 2 and 3 are Pd and Qd
load_mask = network.buses[:, 2] != 0
n_loads = int(load_mask.sum())

# Open load scenarios
loadset = ds.dataset(load_file, format="parquet")

def read_scenario(scenario):
    table = loadset.to_table(
        columns=["p_mw", "q_mvar"],
        filter=ds.field("scenario") == scenario,
    )
    if table.num_rows != n_loads:
        raise ValueError(f"Scenario {scenario}: expected {n_loads} loads, found {table.num_rows}")
    return table["p_mw"].to_numpy(), table["q_mvar"].to_numpy()

# The case is calibrated to scenario 16, so retain that per-bus scaling
p_ref, _ = read_scenario(REFERENCE_SCENARIO)
p, q = read_scenario(SCENARIO)
scale = network.buses[load_mask, 2] / p_ref

# Apply scenario directly to the network arrays
network.buses[load_mask, 2] = p * scale
network.buses[load_mask, 3] = q * scale

print(f"Scenario {SCENARIO}")
print(f"Load buses: {n_loads}")
print(f"Total P load: {network.buses[:, 2].sum():.2f} MW")
print(f"Total Q load: {network.buses[:, 3].sum():.2f} MVAr")


In [ ]:
# Solve OPF
julia = init_julia(max_iter=1000)
opf_result = run_opf(network, julia)

opf_result
